
## Evaluation Metrics
### Shared Evaluation Pipeline for All Models

---

This notebook implements all evaluation metrics from scratch using only NumPy.
All team members import and use these functions to evaluate their models.

**Metrics Implemented:**
1. Accuracy
2. Precision
3. Recall
4. F1-Score
5. Confusion Matrix
6. Classification Report
7. Confusion Matrix Plot

---


In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Confusion Matrix
### Mathematical Formulation

A confusion matrix summarizes the performance of a classification model
by counting correct and incorrect predictions for each class.

**For Binary Classification:**

|                | Predicted Positive | Predicted Negative |
|----------------|-------------------|-------------------|
| Actual Positive | TP (True Positive) | FN (False Negative) |
| Actual Negative | FP (False Positive) | TN (True Negative) |

Where:
- TP → model correctly predicted positive class
- TN → model correctly predicted negative class
- FP → model predicted positive but actual was negative (Type I Error)
- FN → model predicted negative but actual was positive (Type II Error)

**Why is the Confusion Matrix important?**
> Accuracy alone can be misleading for imbalanced datasets.
> MNIST digit 1 has only ~10% of samples — a model that always predicts 0
> would get 90% accuracy but never detect digit 1.
> The confusion matrix reveals this by showing TP, TN, FP, FN separately.

In [2]:
def confusion_matrix(y_true, y_pred):
    """
    Computes the confusion matrix manually.

    Parameters:
        y_true : actual labels      (n_samples,)
        y_pred : predicted labels   (n_samples,)

    Returns:
        cm : confusion matrix (2x2) numpy array
             [[TN, FP],
              [FN, TP]]
    """
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))

    cm = np.array([[TN, FP],
                   [FN, TP]])

    return cm

## Step 2: Accuracy
### Mathematical Formulation

Accuracy measures the fraction of correctly classified samples.

**Formula:**
> Accuracy = (TP + TN) / (TP + TN + FP + FN)

**Intuition:**
> Out of all predictions, how many were correct?

**Limitation:**
> Accuracy is misleading for imbalanced datasets.
> Example: If 90% of samples are class 0, a model that always predicts 0
> gets 90% accuracy but never detects class 1.
> This is why we also compute Precision, Recall and F1-Score.

In [3]:
def accuracy(y_true, y_pred):
    """
    Computes accuracy manually.

    Formula: (TP + TN) / (TP + TN + FP + FN)

    Parameters:
        y_true : actual labels      (n_samples,)
        y_pred : predicted labels   (n_samples,)

    Returns:
        accuracy : float between 0 and 1
    """
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

    acc = (TP + TN) / (TP + TN + FP + FN)

    return acc

## Step 3: Precision
### Mathematical Formulation

Precision measures how many of the predicted positives are actually positive.

**Formula:**
> Precision = TP / (TP + FP)

**Intuition:**
> Out of all samples the model predicted as positive,
> how many were actually positive?

**When is Precision important?**
> When the cost of False Positives is high.
> Example: Spam detection — we don't want to mark legitimate emails as spam.

**Edge case:**
> If the model never predicts positive (TP + FP = 0),
> precision is undefined. We return 0 in this case.

In [4]:
def precision(y_true, y_pred):
    """
    Computes precision manually.

    Formula: TP / (TP + FP)

    Parameters:
        y_true : actual labels      (n_samples,)
        y_pred : predicted labels   (n_samples,)

    Returns:
        precision : float between 0 and 1
    """
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

    # Avoid division by zero
    if (TP + FP) == 0:
        return 0.0

    prec = TP / (TP + FP)

    return prec

## Step 4: Recall
### Mathematical Formulation

Recall measures how many of the actual positives were correctly detected.

**Formula:**
> Recall = TP / (TP + FN)

**Intuition:**
> Out of all samples that are actually positive,
> how many did the model correctly identify?

**When is Recall important?**
> When the cost of False Negatives is high.
> Example: Disease detection — we don't want to miss a sick patient.
> In our case: we don't want to miss digit 1 samples.

**Precision vs Recall tradeoff:**
> Increasing Precision usually decreases Recall and vice versa.
> A model that is very strict about predicting positive → high Precision, low Recall.
> A model that predicts positive for everything → high Recall, low Precision.
> F1-Score balances both.

**Edge case:**
> If there are no actual positives (TP + FN = 0),
> recall is undefined. We return 0 in this case.

In [5]:
def recall(y_true, y_pred):
    """
    Computes recall manually.

    Formula: TP / (TP + FN)

    Parameters:
        y_true : actual labels      (n_samples,)
        y_pred : predicted labels   (n_samples,)

    Returns:
        recall : float between 0 and 1
    """
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

    # Avoid division by zero
    if (TP + FN) == 0:
        return 0.0

    rec = TP / (TP + FN)

    return rec

## Step 5: F1-Score
### Mathematical Formulation

F1-Score is the harmonic mean of Precision and Recall.
It balances both metrics into a single score.

**Formula:**
> F1 = 2 * (Precision * Recall) / (Precision + Recall)

**Why harmonic mean and not regular mean?**
> Regular mean of Precision=1.0 and Recall=0.0 would give 0.5 — misleading.
> Harmonic mean of Precision=1.0 and Recall=0.0 gives 0.0 — correct!
> Harmonic mean penalizes extreme values and rewards balance.

**Interpretation:**
- F1 = 1.0 → perfect precision and recall
- F1 = 0.0 → either precision or recall is 0
- F1 = 0.9+ → excellent model
- F1 = 0.7-0.9 → good model
- F1 < 0.7 → model needs improvement

**When is F1-Score important?**
> When dataset is imbalanced — like our MNIST binary task
> where only ~10% of samples are digit 1.
> F1-Score is the most reliable single metric for imbalanced datasets.

**Edge case:**
> If both Precision and Recall are 0,
> F1 is undefined. We return 0 in this case.

In [6]:
def f1_score(y_true, y_pred):
    """
    Computes F1-Score manually.

    Formula: 2 * (Precision * Recall) / (Precision + Recall)

    Parameters:
        y_true : actual labels      (n_samples,)
        y_pred : predicted labels   (n_samples,)

    Returns:
        f1 : float between 0 and 1
    """
    prec = precision(y_true, y_pred)
    rec  = recall(y_true, y_pred)

    # Avoid division by zero
    if (prec + rec) == 0:
        return 0.0

    f1 = 2 * (prec * rec) / (prec + rec)

    return f1

## Step 6: Classification Report
> A comprehensive summary of all metrics in one clean table.
> Prints Accuracy, Precision, Recall and F1-Score together.

In [7]:
def classification_report(y_true, y_pred, model_name="Model"):
    """
    Prints a comprehensive classification report.
    Includes Accuracy, Precision, Recall and F1-Score.

    Parameters:
        y_true     : actual labels      (n_samples,)
        y_pred     : predicted labels   (n_samples,)
        model_name : name of the model  (default = "Model")
    """
    acc  = accuracy(y_true, y_pred)
    prec = precision(y_true, y_pred)
    rec  = recall(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    cm   = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

    print("="*50)
    print(f" Classification Report — {model_name}")
    print("="*50)
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print("-"*50)
    print(f"  TP : {TP}  |  FP : {FP}")
    print(f"  FN : {FN}  |  TN : {TN}")
    print("="*50)

    return acc, prec, rec, f1

## Step 7: Confusion Matrix Plot
> Visualizes the confusion matrix as a heatmap.
> Makes it easy to see where the model is making mistakes.

**How to read the plot:**
- Top-left  (TN) → correctly predicted negative (not digit 1)
- Top-right (FP) → incorrectly predicted positive (false alarm)
- Bottom-left (FN) → missed positive (missed digit 1)
- Bottom-right (TP) → correctly predicted positive (detected digit 1)

**What we want:**
> High values on the diagonal (TN and TP) — correct predictions
> Low values off the diagonal (FP and FN) — incorrect predictions

In [8]:
def plot_confusion_matrix(y_true, y_pred, model_name="Model"):
    """
    Plots the confusion matrix as a heatmap.

    Parameters:
        y_true     : actual labels      (n_samples,)
        y_pred     : predicted labels   (n_samples,)
        model_name : name of the model  (default = "Model")
    """
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))

    # Plot heatmap manually
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im)

    # Add labels
    classes = ['Not Digit 1 (0)', 'Digit 1 (1)']
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    # Add text annotations
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black",
                   fontsize=14, fontweight='bold')

    ax.set_ylabel('Actual Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()

## Step 8: Final Evaluation Pipeline
> This is the main function that wraps all evaluation steps into one single call.
> Each team member will use this function to evaluate their model.

**Usage:**
```python
evaluate(y_true, y_pred, model_name="My Model")
```

In [9]:
def evaluate(y_true, y_pred, model_name="Model"):
    """
    Full evaluation pipeline.
    Computes all metrics and plots confusion matrix in one call.

    Parameters:
        y_true     : actual labels      (n_samples,)
        y_pred     : predicted labels   (n_samples,)
        model_name : name of the model  (default = "Model")

    Returns:
        results : dictionary containing all metrics
    """
    # Print classification report
    acc, prec, rec, f1 = classification_report(y_true, y_pred, model_name)

    # Plot confusion matrix
    plot_confusion_matrix(y_true, y_pred, model_name)

    # Return results as dictionary
    results = {
        'model'    : model_name,
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1_score' : f1
    }

    return results

In [10]:
!git clone https://github.com/AbdelrahmanAdel-1/MNIST-ML-ImageClassifiers.git
import os
os.chdir('MNIST-ML-ImageClassifiers')
!git config --global user.email "23p0144@eng.asu.edu.eg"
!git config --global user.name "AbdelrahmanAdel-1"
!git checkout main
!git pull origin main

Cloning into 'MNIST-ML-ImageClassifiers'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 74 (delta 23), reused 44 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 184.12 KiB | 8.00 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/AbdelrahmanAdel-1/MNIST-ML-ImageClassifiers
 * branch            main       -> FETCH_HEAD
Already up to date.


In [11]:
%%writefile /content/MNIST-ML-ImageClassifiers/evaluation/metrics.py

import numpy as np
import matplotlib.pyplot as plt

def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    cm = np.array([[TN, FP],
                   [FN, TP]])
    return cm

def accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    acc = (TP + TN) / (TP + TN + FP + FN)
    return acc

def precision(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    if (TP + FP) == 0:
        return 0.0
    prec = TP / (TP + FP)
    return prec

def recall(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    if (TP + FN) == 0:
        return 0.0
    rec = TP / (TP + FN)
    return rec

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec  = recall(y_true, y_pred)
    if (prec + rec) == 0:
        return 0.0
    f1 = 2 * (prec * rec) / (prec + rec)
    return f1

def classification_report(y_true, y_pred, model_name="Model"):
    acc  = accuracy(y_true, y_pred)
    prec = precision(y_true, y_pred)
    rec  = recall(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    print("="*50)
    print(f" Classification Report — {model_name}")
    print("="*50)
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print("-"*50)
    print(f"  TP : {TP}  |  FP : {FP}")
    print(f"  FN : {FN}  |  TN : {TN}")
    print("="*50)
    return acc, prec, rec, f1

def plot_confusion_matrix(y_true, y_pred, model_name="Model"):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    im  = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im)
    classes    = ['Not Digit 1 (0)', 'Digit 1 (1)']
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black",
                   fontsize=14, fontweight='bold')
    ax.set_ylabel('Actual Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

def evaluate(y_true, y_pred, model_name="Model"):
    acc, prec, rec, f1 = classification_report(y_true, y_pred, model_name)
    plot_confusion_matrix(y_true, y_pred, model_name)
    results = {
        'model'    : model_name,
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1_score' : f1
    }
    return results

Writing /content/MNIST-ML-ImageClassifiers/evaluation/metrics.py


In [12]:
!touch /content/MNIST-ML-ImageClassifiers/evaluation/__init__.py

In [13]:
!git add .
!git commit -m "feat: add shared evaluation metrics pipeline"
!git push origin main

[main b1bbcad] feat: add shared evaluation metrics pipeline
 2 files changed, 98 insertions(+)
 create mode 100644 evaluation/__init__.py
 create mode 100644 evaluation/metrics.py
fatal: could not read Username for 'https://github.com': No such device or address
